# Best model — E06 Random Forest

Notebook này chọn mô hình bằng số liệu CV và Validation đã lưu, không xem Test để chọn hoặc điều chỉnh mô hình. Sau khi xác nhận cấu hình E06, notebook fit lại trên Train + Validation rồi đánh giá Test.

File HTML đi kèm được xuất trực tiếp từ notebook đã chạy bằng nbconvert, vì vậy HTML thể hiện đúng code và output trong notebook.

In [1]:
from pathlib import Path
from statistics import mean, stdev
from html import escape
import hashlib
import time

import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Alignment, Border, Font, Side
from openpyxl.utils import get_column_letter
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

OUTPUT_DIR = Path(r"C:\homework\Machine Learning\SGU26K2_MayHoc_DCT124C3_Nhom4_RT-IoT2022\final_IoT\sample_project\result\exps\best model")
PROJECT_ROOT = OUTPUT_DIR.parents[2]
RESULT_ROOT = PROJECT_ROOT / "result"
DATA_DIR = PROJECT_ROOT / "data" / "splitted data"
SELECTION_LOG = RESULT_ROOT / "exps" / "Experiment_Log.xlsx"
HTML_PATH = OUTPUT_DIR / "Best_Model_Test_Evaluation.html"
EXCEL_PATH = OUTPUT_DIR / "Best_Model_Test_Evaluation.xlsx"
TARGET = "Attack_type"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for required in (SELECTION_LOG, DATA_DIR / "train.csv", DATA_DIR / "validation.csv", DATA_DIR / "test.csv"):
    if not required.is_file():
        raise FileNotFoundError(required)
print("Project:", PROJECT_ROOT)
print("Output folder:", OUTPUT_DIR)

Project: C:\homework\Machine Learning\SGU26K2_MayHoc_DCT124C3_Nhom4_RT-IoT2022\final_IoT\sample_project
Output folder: C:\homework\Machine Learning\SGU26K2_MayHoc_DCT124C3_Nhom4_RT-IoT2022\final_IoT\sample_project\result\exps\best model


In [2]:
# Freeze selection using only saved train-CV and Validation scores.
log = load_workbook(SELECTION_LOG, data_only=True, read_only=True).active
candidates = []
for row in range(2, log.max_row + 1):
    folds_f1 = [float(log.cell(row, col).value) for col in (6, 7, 8)]
    folds_recall = [float(log.cell(row, col).value) for col in (10, 11, 12)]
    candidates.append({
        "Experiment": log.cell(row, 1).value,
        "Model": log.cell(row, 3).value,
        "CV Macro-F1": mean(folds_f1),
        "CV Macro-F1 SD": stdev(folds_f1),
        "CV Macro-Recall": mean(folds_recall),
        "CV Macro-Recall SD": stdev(folds_recall),
        "Validation Macro-F1": float(log.cell(row, 14).value),
        "Validation Macro-Recall": float(log.cell(row, 15).value),
    })
selection = pd.DataFrame(candidates)
ranked = selection.sort_values(
    ["CV Macro-F1", "Validation Macro-F1", "Validation Macro-Recall"],
    ascending=False,
    kind="stable",
).reset_index(drop=True)
selected = ranked.iloc[0]
if not str(selected["Model"]).startswith("E06"):
    raise AssertionError(f"Top CV/Validation result no longer selects E06: {selected['Model']}")
# E06 appears in exp1 and exp4 with identical configuration and scores.
e06_rows = selection[selection["Model"].astype(str).str.startswith("E06")]
if len(e06_rows) != 2 or not np.allclose(e06_rows["CV Macro-F1"], selected["CV Macro-F1"]):
    raise AssertionError("Expected the two recorded E06 entries to be equivalent.")
print("Selected before Test evaluation:", selected["Experiment"], "—", selected["Model"])
print("CV Macro-F1 mean ± sample SD:", f"{selected['CV Macro-F1']:.6f} ± {selected['CV Macro-F1 SD']:.6f}")
print("CV Macro-Recall:", f"{selected['CV Macro-Recall']:.6f} ± {selected['CV Macro-Recall SD']:.6f}")
print("Validation Macro-F1 / Macro-Recall:", f"{selected['Validation Macro-F1']:.6f} / {selected['Validation Macro-Recall']:.6f}")
display(ranked.head(5).style.format({column: "{:.6f}" for column in ranked.select_dtypes(include="number").columns}))

Selected before Test evaluation: exp1 — E06 — Random Forest
CV Macro-F1 mean ± sample SD: 0.966133 ± 0.006280
CV Macro-Recall: 0.959367 ± 0.018238
Validation Macro-F1 / Macro-Recall: 0.980809 / 0.970040


,Experiment,Model,CV Macro-F1,CV Macro-F1 SD,CV Macro-Recall,CV Macro-Recall SD,Validation Macro-F1,Validation Macro-Recall
0,exp1,E06 — Random Forest,0.966133,0.006280,0.959367,0.018238,0.980809,0.970040
1,exp4,E06 — Random Forest,0.966133,0.006280,0.959367,0.018238,0.980809,0.970040
2,exp2,E08 — Random Forest,0.965800,0.009226,0.960333,0.020847,0.974980,0.970864
3,baseline,E05 — Random Forest,0.965167,0.004053,0.959333,0.019635,0.980974,0.967969
4,exp4,E07 — Decision Tree,0.961300,0.008802,0.960267,0.020996,0.979412,0.967153


In [3]:
# Use Train + Validation for the final fit, keep Test isolated until prediction.
train = pd.read_csv(DATA_DIR / "train.csv")
validation = pd.read_csv(DATA_DIR / "validation.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
if any(TARGET not in frame.columns for frame in (train, validation, test)):
    raise ValueError(f"Required target column {TARGET!r} is missing.")
if list(train.columns) != list(validation.columns) or list(train.columns) != list(test.columns):
    raise ValueError("Train, Validation, and Test columns/order do not match.")
if train.isna().any().any() or validation.isna().any().any() or test.isna().any().any():
    raise ValueError("Missing values found, E06 pipeline does not impute values.")

X_dev = pd.concat([train.drop(columns=[TARGET]), validation.drop(columns=[TARGET])], ignore_index=True)
y_dev = pd.concat([train[TARGET], validation[TARGET]], ignore_index=True)
X_test = test.drop(columns=[TARGET])[X_dev.columns]
y_test = test[TARGET]
if set(y_dev.unique()) != set(y_test.unique()):
    raise ValueError("Test class labels do not match Train + Validation labels.")

numeric_columns = X_dev.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_columns = [column for column in X_dev.columns if column not in numeric_columns]
preprocessor = ColumnTransformer(
    transformers=[("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_columns)],
    remainder="passthrough",
)
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight="balanced_subsample",
        n_jobs=-1,
    )),
])
start = time.perf_counter()
model.fit(X_dev, y_dev)
fit_seconds = time.perf_counter() - start
print(f"Final fit: {len(X_dev):,} development samples, {len(X_test):,} test samples, {len(numeric_columns)} numeric and {len(categorical_columns)} categorical features.")
print(f"Fit time: {fit_seconds:.3f} seconds")

Final fit: 100,222 development samples, 17,687 test samples, 80 numeric and 2 categorical features.
Fit time: 10.401 seconds


In [4]:
# Test is evaluated only after model selection and final fitting are fixed.
start = time.perf_counter()
y_test_pred = model.predict(X_test)
predict_seconds = time.perf_counter() - start
metrics = {
    "Accuracy": accuracy_score(y_test, y_test_pred),
    "Macro-F1": f1_score(y_test, y_test_pred, average="macro", zero_division=0),
    "Macro-Recall": recall_score(y_test, y_test_pred, average="macro", zero_division=0),
    "Macro-Precision": precision_score(y_test, y_test_pred, average="macro", zero_division=0),
    "Weighted-F1": f1_score(y_test, y_test_pred, average="weighted", zero_division=0),
}
labels = sorted(y_dev.unique())
report = classification_report(y_test, y_test_pred, labels=labels, output_dict=True, zero_division=0)
per_class = pd.DataFrame([
    {"Class": label, "Precision": report[label]["precision"], "Recall": report[label]["recall"],
     "F1": report[label]["f1-score"], "Support": int(report[label]["support"])}
    for label in labels
])
confusion = confusion_matrix(y_test, y_test_pred, labels=labels)
worst_f1 = per_class.sort_values(["F1", "Support"]).iloc[0]
worst_recall = per_class.sort_values(["Recall", "Support"]).iloc[0]
print("Test metrics:")
display(pd.DataFrame([metrics]).T.rename(columns={0: "Test score"}).style.format("{:.6f}"))
print("Lowest class F1:", worst_f1["Class"], f"{worst_f1['F1']:.6f}", "support", int(worst_f1["Support"]))
print("Lowest class recall:", worst_recall["Class"], f"{worst_recall['Recall']:.6f}", "support", int(worst_recall["Support"]))
display(per_class.style.format({"Precision": "{:.6f}", "Recall": "{:.6f}", "F1": "{:.6f}"}))

correct_count = int((y_test.astype(str).to_numpy() == pd.Series(y_test_pred).astype(str).to_numpy()).sum())
majority_class = pd.Series(y_test).value_counts().index[0]
majority_share = float((pd.Series(y_test) == majority_class).mean())
perfect_classes = int((per_class["F1"] == 1.0).sum())
strong_classes = int((per_class["F1"] >= 0.99).sum())
display(Markdown(
    f"Nhận xét: Trên {len(y_test):,} mẫu Test, mô hình dự đoán đúng {correct_count:,} mẫu và sai {len(y_test) - correct_count}. "
    f"Macro-F1 đạt {metrics['Macro-F1']:.4f}, Macro-Recall đạt {metrics['Macro-Recall']:.4f}, "
    f"đây là hai chỉ số phản ánh kết quả trung bình giữa 12 lớp.\n\n"
    f"Kết quả theo lớp tốt ở phần lớn nhãn: {perfect_classes} lớp có F1 bằng 1.000 và "
    f"{strong_classes} lớp có F1 từ 0.990 trở lên. Tuy vậy, Metasploit_Brute_Force_SSH chỉ có "
    f"5 mẫu, mô hình nhận đúng 3 mẫu (Recall {worst_recall['Recall']:.2f}, F1 {worst_recall['F1']:.2f}). "
    f"DDOS_Slowloris cũng thấp hơn các lớp còn lại (F1 {per_class.loc[per_class['Class'] == 'DDOS_Slowloris', 'F1'].iloc[0]:.3f}, "
    f"support {int(per_class.loc[per_class['Class'] == 'DDOS_Slowloris', 'Support'].iloc[0])}).\n\n"
    f"DOS_SYN_Hping chiếm {majority_share:.1%} số mẫu Test. Vì vậy Accuracy/Weighted-F1 cao hơn Macro-F1, "
    "nên xem Macro-F1, Macro-Recall và bảng theo lớp để không bỏ qua nhãn ít mẫu. "
    "Tập Test chỉ dùng cho đánh giá cuối, không dùng để chọn hay điều chỉnh mô hình."
))


Test metrics:


,Test score
Accuracy,0.998813
Macro-F1,0.974680
Macro-Recall,0.960869
Macro-Precision,0.996890
Weighted-F1,0.998797


Lowest class F1: Metasploit_Brute_Force_SSH 0.750000 support 5
Lowest class recall: Metasploit_Brute_Force_SSH 0.600000 support 5


,Class,Precision,Recall,F1,Support
0,ARP_poisioning,0.988725,0.997375,0.993031,1143
1,DDOS_Slowloris,0.986842,0.949367,0.967742,79
2,DOS_SYN_Hping,1.000000,1.000000,1.000000,13514
3,MQTT_Publish,1.000000,1.000000,1.000000,622
4,Metasploit_Brute_Force_SSH,1.000000,0.600000,0.750000,5
5,NMAP_FIN_SCAN,1.000000,1.000000,1.000000,4
6,NMAP_OS_DETECTION,1.000000,1.000000,1.000000,300
7,NMAP_TCP_scan,1.000000,1.000000,1.000000,150
8,NMAP_UDP_SCAN,0.989744,0.994845,0.992288,388
9,NMAP_XMAS_TREE_SCAN,1.000000,0.996678,0.998336,301


Nhận xét: Trên 17,687 mẫu Test, mô hình dự đoán đúng 17,666 mẫu và sai 21. Macro-F1 đạt 0.9747, Macro-Recall đạt 0.9609, đây là hai chỉ số phản ánh kết quả trung bình giữa 12 lớp.

Kết quả theo lớp tốt ở phần lớn nhãn: 6 lớp có F1 bằng 1.000 và 10 lớp có F1 từ 0.990 trở lên. Tuy vậy, Metasploit_Brute_Force_SSH chỉ có 5 mẫu, mô hình nhận đúng 3 mẫu (Recall 0.60, F1 0.75). DDOS_Slowloris cũng thấp hơn các lớp còn lại (F1 0.968, support 79).

DOS_SYN_Hping chiếm 76.4% số mẫu Test. Vì vậy Accuracy/Weighted-F1 cao hơn Macro-F1, nên xem Macro-F1, Macro-Recall và bảng theo lớp để không bỏ qua nhãn ít mẫu. Tập Test chỉ dùng cho đánh giá cuối, không dùng để chọn hay điều chỉnh mô hình.

In [5]:
# Save a compact, Test-only Excel sheet and the detailed HTML report.
excel = Workbook()
ws = excel.active
ws.title = "Kết quả Test"
ws.append(["KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST"])
ws.append(["Mô hình", "E06 — Random Forest"])
ws.append(["Số mẫu Test", len(X_test)])
ws.append([])
ws.append(["Chỉ số tổng thể trên Test"])
ws.append(["Metric", "Score"])
for name, value in metrics.items():
    ws.append([name, float(value)])

ws.append([])
ws.append(["Kết quả theo lớp trên Test"])
ws.append(["Class", "Precision", "Recall", "F1", "Support"])
for row in per_class.itertuples(index=False, name=None):
    ws.append(list(row))
ws.append(["Macro avg", report["macro avg"]["precision"], report["macro avg"]["recall"], report["macro avg"]["f1-score"], int(report["macro avg"]["support"])])
ws.append(["Weighted avg", report["weighted avg"]["precision"], report["weighted avg"]["recall"], report["weighted avg"]["f1-score"], int(report["weighted avg"]["support"])])

ws.append([])
ws.append(["Confusion matrix — hàng: nhãn thật, cột: nhãn dự đoán"])
ws.append(["Actual \\ Predicted"] + labels)
for label, values in zip(labels, confusion):
    ws.append([label] + [int(value) for value in values])

line = Side(style="thin", color="D9D9D9")
section_rows = {1, 5, 14, 30}
header_rows = {6, 15, 30}
for row in ws.iter_rows():
    for cell in row:
        cell.font = Font(name="Calibri", size=10, bold=(cell.row in section_rows or cell.row in header_rows), color="000000")
        cell.border = Border(left=line, right=line, top=line, bottom=line)
        cell.alignment = Alignment(vertical="top", wrap_text=True)
for column, width in {"A": 36, "B": 18, "C": 18, "D": 18, "E": 14}.items():
    ws.column_dimensions[column].width = width
for column in range(6, len(labels) + 2):
    ws.column_dimensions[get_column_letter(column)].width = 22
for row in range(7, 12):
    ws.cell(row, 2).number_format = "0.000000"
for row in range(16, 16 + len(labels) + 2):
    for column in range(2, 5):
        ws.cell(row, column).number_format = "0.000000"
ws.sheet_view.showGridLines = True
ws.freeze_panes = None
excel.save(EXCEL_PATH)

print("Saved Test-only Excel:", EXCEL_PATH)
print("Workbook includes overall Test metrics, per-class results, and the Test confusion matrix.")


Saved Test-only Excel: C:\homework\Machine Learning\SGU26K2_MayHoc_DCT124C3_Nhom4_RT-IoT2022\final_IoT\sample_project\result\exps\best model\Best_Model_Test_Evaluation.xlsx
Workbook includes overall Test metrics, per-class results, and the Test confusion matrix.
